# RG3 F1: 전문가 동의 규칙

94번의5개 앙상블 전문가(ref가 평가구간을 제외한 저장 모델)의 점수만 활용. 각 전문가 내부 OOF 정상 분위수로 threshold 결정. single/AND/OR/2of3규칙과 분위수는 사전 고정. 내부 F1·동률탐지율·동률오탐순 선택. 주결과는 inner_F1_selected_consensus, 가족별 선택도 보고. 많은 규칙 탐색이 학습 내부에 과적합할 수 있으므로 전체 외부 평가 성적과 구분.

RG3 RH 정상601·가스19/같은7개 관측구간. 평가 구간을 모델/조건부순위/정상기준/규칙/임계값 학습에서 제외. 알려진 자료 개발용 중첩 검증이며 새 독립 검증 아님. 단일 전문가 자체도 여러 모델 평균 앙상블.


In [1]:
from pathlib import Path
import numpy as np,pandas as pd
from itertools import combinations,product
from sklearn.metrics import confusion_matrix
root=Path.cwd()
if not (root/'data').exists():root=root.parent
t=root/'reports/tables'
z=np.load(t/'rg3_F1_boost_fold_cache.npz',allow_pickle=False)
y=z['labels'];groups=z['groups'];times=z['timestamps'];blocks=np.unique(groups)
assert len(y)==620 and y.sum()==19 and (y==0).sum()==601
cache={tuple(z[k].astype(int)):z['scores_'+k.split('_')[1]] for k in z.files if k.startswith('ref_')}
cols=[0,1,2,4,5]
def get(ref,q):
 assert set(groups[ref]).isdisjoint(groups[q])
 return cache[tuple(ref)][q][:,cols]
def stats(yy,p):
 tn,fp,fn,tp=confusion_matrix(yy,p,labels=[0,1]).ravel()
 return dict(TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0,recall=tp/(tp+fn) if tp+fn else 0)
records=[];choices=[];search=[]
budgets=[.01,.025,.05,.1]
rules=[]
for i in range(5):
 for rate in budgets:rules.append(('single',(i,),(rate,),1))
for ids in combinations(range(5),2):
 for rates in product(budgets,repeat=2):
  rules.extend([('AND',ids,rates,2),('OR',ids,rates,1)])
for ids in combinations(range(5),3):
 for rates in product([.025,.05,.1],repeat=3):rules.append(('vote2of3',ids,rates,2))
for b in blocks:
 q=np.flatnonzero(groups==b);ref=np.flatnonzero(groups!=b)
 out=get(ref,q);inner=np.full((len(ref),5),np.nan)
 for ib in np.unique(groups[ref]):
  mask=groups[ref]==ib;inner[mask]=get(ref[~mask],ref[mask])
 cuts={r:np.quantile(inner[y[ref]==0],1-r,axis=0,method='higher') for r in budgets}
 candidates=[]
 for family,ids,rates,votes in rules:
  ip=np.column_stack([inner[:,i]>cuts[r][i] for i,r in zip(ids,rates)]).sum(1)>=votes
  op=np.column_stack([out[:,i]>cuts[r][i] for i,r in zip(ids,rates)]).sum(1)>=votes
  result=stats(y[ref],ip)
  candidates.append((result['F1'],result['recall'],-result['FP'],family,ids,rates,votes,op))
  search.append(dict(block=int(b),family=family,experts=','.join(map(str,ids)),budgets=','.join(map(str,rates)),votes=votes,**result))
 best=max(candidates,key=lambda a:a[:3])
 named=[('inner_F1_selected_consensus',best)]
 for family in ['single','AND','OR','vote2of3']:
  named.append((family+'_inner_F1_selected',max((c for c in candidates if c[3]==family),key=lambda a:a[:3])))
 for name,c in named:
  for j,i in enumerate(q):records.append(dict(model=name,TimeStamp=times[i],block=int(b),actual=int(y[i]),prediction=int(c[7][j])))
 choices.append(dict(block=int(b),family=best[3],experts=','.join(map(str,best[4])),budgets=','.join(map(str,best[5])),votes=best[6],inner_F1=best[0],inner_recall=best[1]))
 print('Finished consensus block',b,flush=True)
p=pd.DataFrame(records);m=pd.DataFrame([dict(model=name,**stats(g.actual,g.prediction)) for name,g in p.groupby('model')])
for obj,name in [(p,'predictions'),(m,'metrics'),(pd.DataFrame(choices),'inner_choices'),(pd.DataFrame(search),'inner_search')]:obj.to_csv(t/f'rg3_consensus_gates_{name}.csv',index=False,encoding='utf-8-sig')
print(m.to_string(index=False))
print('Rule/threshold selection uses inner labels only; known-data development evaluation.')


Finished consensus block 1


Finished consensus block 2


Finished consensus block 3


Finished consensus block 4


Finished consensus block 5


Finished consensus block 6


Finished consensus block 7


                      model  TP  FP  FN  TN       F1   recall
      AND_inner_F1_selected   3  17  16 584 0.153846 0.157895
       OR_inner_F1_selected   7  48  12 553 0.189189 0.368421
inner_F1_selected_consensus   3  29  16 572 0.117647 0.157895
   single_inner_F1_selected   4  55  15 546 0.102564 0.210526
 vote2of3_inner_F1_selected   4  37  15 564 0.133333 0.210526
Rule/threshold selection uses inner labels only; known-data development evaluation.
